## Notebook to evaluate the effect of added Noise on standard and robust clustering-algorithms
### Goal of this notebook to compare standard and robust clustering algorithms on different forms and intensities of noise

#### Generating data


In [1]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=1000, centers=3, n_features=10, cluster_std=1.0, random_state=42)



#### Plot the original dataset

In [3]:
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

df = X,y

pca = PCA(n_components=2)
components = pca.fit_transform(X)

fig = px.scatter(components, x=0, y=1, color=y)
fig.show()





#### Function to add noise
Die Funktion erzeugt Ausreißer, indem sie zusätzliche Rauschpunkte zum bestehenden Datensatz hinzufügt, anstatt vorhandene Datenpunkte zu verändern. Die Anzahl dieser Ausreißer wird prozentual zur ursprünglichen Datensatzgröße festgelegt. Bei einem Rauschanteil von 10 % werden beispielsweise zu 1000 ursprünglichen Datenpunkten 100 zusätzliche Ausreißer erzeugt.

Zur Generierung dieser Ausreißer wird zunächst der Wertebereich der vorhandenen Daten bestimmt. Anschließend werden neue Punkte zufällig so erzeugt, dass sie außerhalb dieses ursprünglichen Bereichs liegen. Dadurch wird sichergestellt, dass die neuen Punkte nicht innerhalb der bestehenden Cluster liegen und somit tatsächlich als Ausreißer wirken.

Das Ergebnis ist ein erweiterter Datensatz, der neben den ursprünglichen Clusterpunkten auch künstlich erzeugte Störpunkte enthält. Auf diese Weise kann untersucht werden, wie empfindlich Clustering-Verfahren auf zusätzliche Ausreißer reagieren.

In [ ]:
def add_outliers(X, y=None, noise_percent=10, min_distance_factor=2.5, random_state=42):
    """
    Add outlier points that are far away from the cluster center region.
    """
    rng = np.random.default_rng(random_state)

    n_samples, n_features = X.shape
    n_noise = int((noise_percent / 100) * n_samples)

    if n_noise == 0:
        return X.copy(), None if y is None else y.copy(), np.empty((0, n_features))

    center = X.mean(axis=0)
    dists = np.linalg.norm(X - center, axis=1)
    typical_radius = np.percentile(dists, 95)

    noise_points = []
    while len(noise_points) < n_noise:
        candidates = rng.normal(loc=center, scale=X.std(axis=0) * 3, size=(n_noise, n_features))
        candidate_dists = np.linalg.norm(candidates - center, axis=1)

        valid = candidates[candidate_dists > min_distance_factor * typical_radius]
        noise_points.extend(valid.tolist())

    X_noise = np.array(noise_points[:n_noise])
    X_new = np.vstack([X, X_noise])

    if y is not None:
        y_new = np.concatenate([y, np.full(n_noise, -1)])
    else:
        y_new = None

    return X_new, y_new, X_noise

#### Data preparation